In [1]:
import pandas as pd

product_features = pd.read_csv(
    "../data/processed/product_features.csv",
    index_col=0
)

features = [
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count"
]

In [2]:
product_features[features].describe()

,order_frequency,total_quantity,avg_quantity_per_order,related_product_count
count,3922.000000,3922.000000,3922.000000,3922.000000
mean,132.484192,1420.810811,29.417691,1912.547680
std,193.890513,3579.835163,1293.401511,1006.136593
min,1.000000,1.000000,1.000000,0.000000
25%,16.000000,54.000000,2.600000,1103.250000
50%,65.000000,369.500000,5.413199,2162.000000
75%,165.000000,1393.750000,10.229681,2759.750000
max,2198.000000,80995.000000,80995.000000,3562.000000


In [3]:
import numpy as np

X_log = np.log1p(
    product_features[features]
)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_log)

In [5]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

product_features["cluster"] = kmeans.fit_predict(X_scaled)

In [6]:
product_features["cluster"].value_counts().sort_index()

cluster
0    1412
1     419
2    1224
3     867
Name: count, dtype: int64

In [7]:
cluster_summary = product_features.groupby(
    "cluster"
)[features].mean()

cluster_summary

,order_frequency,total_quantity,avg_quantity_per_order,related_product_count
cluster,,,,
0,111.949008,515.193343,5.342067,2285.064448
1,2.959427,28.131265,9.662591,124.968974
2,283.126634,3920.123366,82.895252,2609.807190
3,15.852364,40.310265,2.676782,1185.392157


In [9]:
cluster_names = {
    0: "Active/Regular Products",
    1: "Slow-Moving Products",
    2: "High-Volume Products",
    3: "Low-Movement Products"
}

product_features["cluster_name"] = (
    product_features["cluster"].map(cluster_names)
)

In [10]:
product_features["cluster_name"].value_counts()

cluster_name
Active/Regular Products    1412
High-Volume Products       1224
Low-Movement Products       867
Slow-Moving Products        419
Name: count, dtype: int64

In [11]:
product_features.to_csv(
    "../data/processed/product_clusters_v2.csv",
    index=False
)